# kinapse — Colab quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LilianDenzler/kinapse/blob/main/examples/colab_quickstart.ipynb)

**kinapse** is a modular toolkit for TCR / TCR-pMHC structural analysis. This notebook runs
**pip-only on a plain CPU Colab runtime** — no conda — and walks through the lightweight core:
load & IMGT-number a TCR, read its CDR loops, list the pluggable model registry, grade a model
against ground truth (HQ/MQ/AQ/LQ), and (optionally) score an interface.

> The MD/dynamics and α/β-geometry features need the conda MD stack (bioconda ANARCI) and are
> **not** covered here — see the repo's local notebooks and `docs/` for those.

## 1. Install

kinapse currently lives in a **private** GitHub repo, so the clone needs a GitHub token with
read access. (If you make the repo public, leave the token blank — it clones anonymously.)
Cloning also gives us the small example structure the rest of the notebook uses.

In [ ]:
import os, getpass, subprocess

# Paste a GitHub token (Settings → Developer settings → tokens; needs repo read).
# Leave blank if the repo is public.
TOKEN = getpass.getpass("GitHub token (blank if public): ").strip()
auth = f"{TOKEN}@" if TOKEN else ""
REPO = f"https://{auth}github.com/LilianDenzler/kinapse.git"

if not os.path.isdir("kinapse_repo"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "kinapse_repo"], check=True)

os.environ["ANARCI_CPU"] = "1"                 # force CPU numbering on Colab
DATA = "kinapse_repo/examples/data"            # bundled example structures
print("cloned:", sorted(os.listdir(DATA)))

In [ ]:
# Core + the `structures` extra (biopython, mdtraj, ANARCII numbering, freesasa) — all pip wheels.
%pip install -q "./kinapse_repo[structures]"

## 2. Load & IMGT-number a TCR

`TCR(...)` parses the PDB, numbers the chains with **ANARCII** (`legacy_anarci=False`, the
pip-installable numberer), pairs the α/β chains, and slices the variable domain. The example is
an α/β Fv; the same call works on a full **TCR-pMHC complex** (it finds the TCR and treats the
rest as pMHC). You'll see a few `[info] Numbering …` lines — that's ANARCII, not an error.

In [ ]:
import kinapse
from kinapse.structures import TCR

tcr = TCR(input_pdb=f"{DATA}/example_tcr.pdb", legacy_anarci=False)
pair = tcr.pairs[0]
print("kinapse", getattr(kinapse, "__version__", "?"))
print("paired chains (α, β):", pair.alpha_chain_id, pair.beta_chain_id)
print("# α/β pairs found:", len(tcr.pairs))

In [ ]:
# The six CDR loop sequences (IMGT-defined) — the business end of the receptor
cdr_fr = pair.cdr_fr_sequences()
{k: v for k, v in cdr_fr.items() if "CDR" in k}

## 3. IMGT regions & the variable structure

`CDR_FR_RANGES` are the IMGT numbering ranges kinapse uses to slice CDR/FR regions;
`pair.variable_structure` is a Biopython structure of the Fv (renumbered, chains renamed A/B).

In [ ]:
from kinapse.regions import CDR_FR_RANGES

print("region → IMGT range (first few):")
for name, (lo, hi) in list(CDR_FR_RANGES.items())[:6]:
    print(f"  {name:8s} {lo}–{hi}")

vs = pair.variable_structure
n_res = sum(1 for ch in vs.get_chains() for r in ch if r.id[0] == " ")
print("\nvariable-domain chains:", [c.id for c in vs.get_chains()], "| Cα residues:", n_res)

## 4. The pluggable model registry

Every external model (structure predictors, embedders, scorers, docking, …) is a `RunnerSpec`
in a **uniform registry** — so you can see what's available and add your own with one spec + one
script (see `docs/ADDING_A_MODEL.md`). Listing needs nothing installed.

In [ ]:
from kinapse import runners

for tier in ("scoring", "structure_prediction", "conformer_generation"):
    specs = runners.specs(tier)
    print(f"{tier:22s} ({len(specs)}): " + ", ".join(s.name for s in specs[:6])
          + (" …" if len(specs) > 6 else ""))

## 5. Structural agreement — is a model close to ground truth?

Given a **modelled** complex and its **ground-truth** structure, `structural_agreement` IMGT-numbers
both, finds the CDRs, and computes the **Cα iRMSD over the 6 CDR loops** (plus per-CDR RMSDs), then
`assign_tier` maps it to **HQ / MQ / AQ / LQ**. Here we compare the example to *itself* → 0 Å → `HQ`.
On real data you'd pass `structural_agreement(model_pdb, gt_pdb)`.

In [ ]:
from kinapse.benchmarks import structural_agreement, assign_tier

s = structural_agreement(f"{DATA}/example_tcr.pdb", f"{DATA}/example_tcr.pdb")   # model vs GT
print("Cα iRMSD over 6 CDRs:", s["struct__cdr_irmsd"], "Å  → tier", assign_tier(s["struct__cdr_irmsd"]))

# how the tiers map (Cα iRMSD, optional DockQ):
for irmsd, dockq in [(1.5, 0.9), (3.0, 0.6), (4.9, 0.3), (7.0, 0.9)]:
    print(f"  iRMSD={irmsd} Å, DockQ={dockq} → {assign_tier(irmsd, dockq)}")

## 6. (optional) Score an interface with ifscore

kinapse delegates interface scoring to the separate **ifscore** package. Its `geometry_scoring`
metric (BSA / SASA / contacts) runs **in-process with no external tool**, so it works on Colab
after a quick `pip install`. Other scorers (DockQ, PRODIGY, FoldX, GPU predictors, …) need
`ifscore install <name>` and are covered in `docs/scoring.md`.

In [ ]:
%pip install -q "ifscore @ git+https://github.com/LilianDenzler/scoring_functions"

from kinapse import scoring
# score the α–β interface of the example (receptor = chain A, ligand = chain B)
r = scoring.score(f"{DATA}/example_tcr.pdb", rec=["A"], lig=["B"], scorers="geometry_scoring")
{k: r[k] for k in r if k in ("geometry__bsa", "geometry__sasa_complex", "geometry__n_contacts_5A")}

## Next steps

- **Module map:** `kinapse_repo/docs/MODULES.md` — every module, its extra, and status.
- **Add your own model:** `kinapse_repo/docs/ADDING_A_MODEL.md`.
- **Scoring:** `kinapse_repo/docs/scoring.md` (ifscore scorers, incl. GPU predictors).
- **Benchmarking:** `kinapse_repo/docs/benchmarks.md` + `experiments/scorer_benchmarking/`
  (GT-vs-model-vs-negative scorer benchmark, HQ/MQ/AQ/LQ tiers, native vs STCRpy loader).
- **Local notebooks** (need the conda MD stack): `examples/01…07` cover structures, geometry,
  dynamics, scoring, adding a model, and benchmarking.

Install the full stack locally with `./setup.sh` (conda/mamba) — see the repo README.